## Load the corpus and confirm the GPU is usable

In [ ]:
"""Evaluate a strong multilingual cross-encoder as a reranker, on a Kaggle GPU.

A small reranker (mmarco-mMiniLMv2) was measured locally and made retrieval worse:
0.3379 -> 0.2864 at depth 10, and 0.2356 at depth 20. That model is trained on short web
queries and ESMA questions are long legal prose, so the open question is whether the
failure is reranking or that particular reranker. bge-reranker-v2-m3 answers it, and at
3.2 s/pair on a laptop CPU it is a 25 hour run there and minutes here.
"""
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

RETRIEVER = "intfloat/multilingual-e5-small"
RERANKER = "BAAI/bge-reranker-v2-m3"
CHUNK_TOKENS, OVERLAP_TOKENS, MIN_TAIL_TOKENS = 400, 80, 40
DEPTH, K, RRF_K = 20, 10, 60
LANGUAGES = ("en", "nl", "de", "fr")


def find_data() -> Path:
    root = Path("/kaggle/input")
    print("mounted inputs:", [str(p) for p in root.glob("*")], flush=True)
    for candidate in root.rglob("articles.jsonl"):
        return candidate.parent
    raise SystemExit(f"articles.jsonl not found under {root}")


DATA = find_data()
articles = [json.loads(x) for x in (DATA / "articles.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
judgements = [json.loads(x) for x in (DATA / "judgements.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
print(f"articles={len(articles)} judgements={len(judgements)}", flush=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    name = torch.cuda.get_device_name(0)
    try:
        (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
        print(f"device=cuda ({name})", flush=True)
    except Exception as error:
        print(f"GPU {name} unusable ({type(error).__name__}); this run needs a T4", flush=True)
        raise SystemExit(1)
else:
    raise SystemExit("no GPU; set Accelerator to GPU T4 x2 before running")

## Chunk the corpus

In [ ]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())


def ndcg_at_k(retrieved, relevant, k):
    gold = set(relevant)
    if not gold:
        return 0.0
    gain = sum(1.0 / math.log2(i + 1) for i, p in enumerate(retrieved[:k], start=1) if p in gold)
    ideal = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(gold), k) + 1))
    return gain / ideal if ideal else 0.0


def recall_at_k(retrieved, relevant, k):
    gold = set(relevant)
    return sum(1 for p in retrieved[:k] if p in gold) / len(gold) if gold else 0.0


retriever = SentenceTransformer(RETRIEVER, device=device)
tokenizer = retriever.tokenizer


def chunk_text(text, size=CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= size:
        return [text]
    step = size - overlap
    min_tail = max(1, min(MIN_TAIL_TOKENS, size // 4))
    pieces = []
    for start in range(0, len(ids), step):
        window = ids[start:start + size]
        if len(window) < min_tail and pieces:
            break
        pieces.append(tokenizer.decode(window))
        if start + size >= len(ids):
            break
    return pieces


passages = defaultdict(list)
for a in articles:
    for piece in chunk_text(a["text"]):
        passages[a["language"]].append({"celex": a["celex"], "article": a["article"],
                                        "language": a["language"], "text": piece})
print({lang: len(v) for lang, v in passages.items()}, flush=True)

## Build the dense and lexical indexes

In [ ]:
from rank_bm25 import BM25Okapi

matrices, bm25 = {}, {}
for lang in LANGUAGES:
    units = passages[lang]
    matrices[lang] = retriever.encode(["passage: " + u["text"] for u in units], batch_size=128,
                                      convert_to_numpy=True, normalize_embeddings=True,
                                      show_progress_bar=False)
    bm25[lang] = BM25Okapi([tokenize(u["text"]) for u in units])
    print(f"indexed {lang}: {len(units)} passages", flush=True)


def collapse(scored, k):
    best = {}
    for idx, score, lang in scored:
        u = passages[lang][idx]
        key = (u["celex"], u["article"], u["language"])
        if key not in best or score > best[key]:
            best[key] = score
    ordered = sorted(best.items(), key=lambda kv: -kv[1])[:k]
    return [(key, score) for key, score in ordered]


def hybrid(query, lang, depth):
    q = retriever.encode(["query: " + query], convert_to_numpy=True, normalize_embeddings=True)[0]
    dense = matrices[lang] @ q
    d_order = np.argsort(-dense)[:depth * 3]
    lex = bm25[lang].get_scores(tokenize(query))
    l_order = np.argsort(-lex)[:depth * 3]
    dense_run = collapse([(int(i), float(dense[i]), lang) for i in d_order], depth * 2)
    lex_run = collapse([(int(i), float(lex[i]), lang) for i in l_order], depth * 2)
    fused = defaultdict(float)
    for run in (dense_run, lex_run):
        for rank, (key, _s) in enumerate(run, start=1):
            fused[key] += 1.0 / (RRF_K + rank)
    return [k for k, _ in sorted(fused.items(), key=lambda kv: -kv[1])[:depth]]

## Rerank and score

In [ ]:
reranker = CrossEncoder(RERANKER, max_length=512, device=device)
print("reranker loaded", flush=True)

best_text = {}
for lang in LANGUAGES:
    for u in passages[lang]:
        best_text.setdefault((u["celex"], u["article"], u["language"]), u["text"])

rows = defaultdict(lambda: defaultdict(list))
for n, j in enumerate(judgements, 1):
    lang = j["target_language"]
    gold = [tuple(p) for p in j["relevant"]]
    hits = hybrid(j["query"], lang, DEPTH)
    order = [(c, a) for c, a, _l in hits]
    slice_name = j["slice_name"]

    scores = reranker.predict([(j["query"], best_text[h]) for h in hits], batch_size=32,
                              show_progress_bar=False)
    ranked = [(c, a) for (c, a, _l), _s in sorted(zip(hits, scores), key=lambda x: -x[1])]
    top10 = sorted(zip(hits[:K], scores[:K]), key=lambda x: -x[1])

    for name, seq in (("baseline", order), ("rerank_depth20", ranked),
                      ("rerank_depth10", [(c, a) for (c, a, _l), _s in top10])):
        rows[name][slice_name].append(ndcg_at_k(seq, gold, K))
    rows["ceiling"][slice_name].append(recall_at_k(order, gold, DEPTH))
    if n % 100 == 0:
        print(f"  {n}/{len(judgements)}", flush=True)

summary = {}
for name, slices in rows.items():
    every = [v for vals in slices.values() for v in vals]
    summary[name] = {"overall": round(sum(every) / len(every), 4),
                     **{s: round(sum(v) / len(v), 4) for s, v in slices.items()}}
    print(f"{name}: {summary[name]}", flush=True)
Path("/kaggle/working/rerank_results.json").write_text(json.dumps(summary, indent=2))
print("DONE", flush=True)